<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Network Addressing and Routing

So far, you have used the shell to work with files and programs in the environment where your commands run. Working with a Duckiedrone adds another requirement: your development computer must exchange data with a separate computer on the Duckiedrone.

For example, opening a Duckiedrone's dashboard requires your computer to know where to send its request. Being connected to Wi-Fi is only part of the answer: which address belongs to the Duckiedrone, and how should data travel there?

This notebook introduces the information computers use to make those decisions. You will inspect network interfaces and addresses, identify a local subnet, and read a routing table to distinguish destinations reached locally from those reached through a gateway.

The worked examples use a recorded capture from a physical Duckiedrone named `amelia`. It is a reference case, not the name of your Duckiedrone; its addresses describe the network used when the output was recorded, so your setup will probably use different values.

> __Where to run the commands__
>
> These are Linux commands. On an Ubuntu base station, run them in a terminal.
>
> In a Duckietown Workspace terminal, these same commands would describe the development environment's network configuration. Its interfaces and gateway may differ from those of the host machine.
>
> You can complete the following activities using `amelia`'s recorded output below. For an authorized Duckiedrone shell, use [Notebook 13](./13-physical-duckiedrone-ssh-access.ipynb) for a physical Duckiedrone or [Notebook 14](./14-virtual-duckiedrone-connections.ipynb) for a virtual Duckiedrone. Later in this LX, `DUCKIEDRONE_NAME` refers to your Duckiedrone's name.

## Local networks

Suppose you want to open a Duckiedrone's dashboard from your base station (the development computer you use to work with the Duckiedrone). The two computers need a connection over which they can exchange data.

A common arrangement is shown in [Figure 1](#figure-1).

<figure id="figure-1" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base station
      |
      v
    Wi-Fi <- Duckiedrone
      |
      v
    wireless access point
      |
      v
    local network
      |
      v
    router
      |
      v
    other networks (including the internet)
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 1: Local Wi-Fi topology connecting a base station and Duckiedrone.</figcaption>
</figure>

This is an example arrangement, not a required wiring diagram. The access point and router may be parts of the same physical device.

A wireless access point connects Wi-Fi devices to a local network. If you connect the base station with an Ethernet cable instead, a __switch__ can connect it to other devices on that network. A __router__ connects different networks and forwards data between them.

A home Wi-Fi router commonly combines these roles in one box. The distinction matters because exchanging data with a nearby Duckiedrone and reaching a website on the internet do not necessarily use the same path.

Computers send network data in units called __packets__. An __Internet Protocol (IP)__ packet includes source and destination IP addresses, which identify where it comes from and where it is intended to go.

For communication within the local network, packets may pass through an access point or switch without being routed to another network. Consequently, a local Duckiedrone connection can still work when the network's internet connection is unavailable.

__Question:__ If the internet connection in the diagram fails, but the local network keeps working, does that failure alone prevent the base station from communicating with the Duckiedrone?

<details>
<summary>Reveal answer</summary>

No. Both computers are on the local network, so their communication does not require the internet connection. Whether the dashboard actually works also depends on the Duckiedrone and its software; we will investigate those additional requirements later.

</details>

## Interfaces and identifiers

Before choosing where to send a packet, a computer needs a connection through which to send it.

A __network interface__ is such a connection. A computer might have an Ethernet interface for a cable, a Wi-Fi interface for wireless communication, and additional virtual interfaces created by software.

First, identify the computer where the command runs:

```bash
hostname
```

For our example Duckiedrone, the output is:

```shell
amelia
```

This is its __hostname__, a human-readable name. Running `hostname` on your base station or inside your Workspace reports the name of that environment instead. The command also works in a macOS terminal.

### Find the interface with an address

On Linux, `ip` is the command used here to inspect the current network context without changing it. Its `address` subcommand lists interface names and assigned IP addresses, its `link` subcommand lists link-layer information such as interface state and __Media Access Control (MAC)__ address, and its `route` subcommand displays the local routing table. The `-brief` option requests a compact display.

Run:

```bash
ip -brief address
```

The relevant rows from `amelia`'s captured output are:

```shell
lo               UNKNOWN        127.0.0.1/8 ::1/128
eth0             DOWN
wlan0            UP             192.168.1.201/24 metric 600
```

[Table 1](#table-1) summarizes the relevant interface capture.

<table id="table-1" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 1: Relevant interfaces reported by <code>amelia</code>'s address capture.</caption>
  <thead>
    <tr>
      <th>Interface</th>
      <th>What this capture tells us</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>lo</code></td>
      <td>The loopback interface, used for communication within <code>amelia</code> itself.</td>
    </tr>
    <tr>
      <td><code>eth0</code></td>
      <td>The Ethernet interface is down and has no IP address shown; it may be disabled or lack an active link.</td>
    </tr>
    <tr>
      <td><code>wlan0</code></td>
      <td>The Wi-Fi interface is up and has an Internet Protocol version 4 (IPv4) address <code>192.168.1.201</code>.</td>
    </tr>
  </tbody>
</table>

For the physical network connection in this example, focus on `wlan0`. The `/24` beside its address describes the subnet; we will interpret it in the next section. The `metric 600` field is the priority of the prefix route associated with this address; we will return to route priorities later.

An `UP` interface is useful evidence, but it does not establish that a particular remote computer or dashboard is reachable. Conversely, the loopback interface's `UNKNOWN` state is normal here and does not mean it is broken.

Interface names vary between computers. Your Wi-Fi interface could be named differently from `wlan0`.

### Distinguish network and link-layer addresses

The Wi-Fi interface also has a MAC address, used to identify it on its local network connection. A hardware interface usually has a factory-assigned MAC address, but software and Wi-Fi privacy features can change the address that the interface uses.

To inspect link information, run:

```bash
ip -brief link
```

The relevant row from `amelia`'s output is:

```shell
wlan0            UP             dc:a6:32:31:43:ad <BROADCAST,MULTICAST,UP,LOWER_UP>
```

The values after the MAC address are interface flags. [Table 2](#table-2) compares the identifiers for the same interface.

<table id="table-2" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 2: Identifiers for <code>amelia</code>'s Wi-Fi interface.</caption>
  <thead>
    <tr>
      <th>Identifier</th>
      <th><code>amelia</code>'s Wi-Fi interface</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Interface name</td>
      <td><code>wlan0</code></td>
    </tr>
    <tr>
      <td>MAC address</td>
      <td><code>dc:a6:32:31:43:ad</code></td>
    </tr>
    <tr>
      <td>IPv4 address</td>
      <td><code>192.168.1.201</code></td>
    </tr>
  </tbody>
</table>

These identifiers serve different purposes. `wlan0` is the name Linux uses for the interface. Its MAC address identifies it on the local connection. Its IP address is used to address packets, including packets that may need to travel across routers.

For example, a lab network can require the Duckiedrone's Wi-Fi MAC address for registration under its access policy. A program contacting the Duckiedrone normally uses its hostname or IP address instead.

### Recognize addresses that stay inside the computer

`amelia`'s `lo` row contains two loopback addresses:

```shell
127.0.0.1/8 ::1/128
```

`127.0.0.1` is the familiar IPv4 loopback address; `::1` is the Internet Protocol version 6 (IPv6) equivalent. Traffic sent to these addresses stays within the current network environment.

## Subnets and gateways

Knowing `amelia`'s address still leaves a question: which destinations can it reach on the local network, and which require a router?

An __IPv4 address__ contains four decimal groups, each between `0` and `255`. The __subnet prefix__ identifies the network portion of the address.

Consider `amelia`'s Wi-Fi address:

```shell
192.168.1.201/24
```

Each decimal group represents 8 bits. The `/24` says that the first 24 bits (the first three groups in this example) identify the network, as shown in [Figure 2](#figure-2).

<figure id="figure-2" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    192.168.1.201
    |_______| |_|
     network  host
    (24 bits) (8 bits)
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 2: Network and host portions of <code>amelia</code>'s IPv4 address.</figcaption>
</figure>

Here, the subnet is `192.168.1.0/24`. The host portion distinguishes addresses within it. You may also see `/24` expressed as the __subnet mask__ `255.255.255.0`.

Suppose a base station on the same local network has address `192.168.1.42/24`. The two hosts share the local subnet shown in [Figure 3](#figure-3).

<figure id="figure-3" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base station (192.168.1.42/24)
      |
      v
    local subnet (192.168.1.0/24) <- amelia (192.168.1.201/24)
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 3: Two hosts on the same Wi-Fi subnet.</figcaption>
</figure>

Both addresses are on the same local subnet, so each host can select a direct route to the other without using a gateway; successful communication still depends on the link and access policy. Now consider another device with destination at, e.g., `192.168.2.42`. Its address is not in this Wi-Fi subnet, so both the base station and `amelia` would need a route beyond that local connection.

A __gateway__ is a router used as the next step toward a destination. The __default gateway__ handles destinations for which the computer has no more specific route. [Table 3](#table-3) compares example destinations with `amelia`'s Wi-Fi subnet.

<table id="table-3" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 3: Example destinations relative to <code>amelia</code>'s Wi-Fi subnet.</caption>
  <thead>
    <tr>
      <th>Destination from <code>amelia</code></th>
      <th>Relationship to <code>192.168.1.0/24</code></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>192.168.1.42</code></td>
      <td>Inside the Wi-Fi subnet</td>
    </tr>
    <tr>
      <td><code>192.168.1.77</code></td>
      <td>Inside the Wi-Fi subnet</td>
    </tr>
    <tr>
      <td><code>192.168.2.42</code></td>
      <td>Outside the Wi-Fi subnet</td>
    </tr>
  </tbody>
</table>

Comparing the first three groups works here because the prefix is `/24`. Other prefix lengths divide the address differently. The subnet explains the local address range, but to see where Linux actually sends packets, we need its _routing table_.

## Network names

The Domain Name System (DNS) maps names to numeric addresses, so remembering an address for every Duckiedrone would become inconvenient.

A __hostname lookup__ (also called __name resolution__) finds IP addresses associated with a name. DNS provides this mapping for names such as `duckietown.com`. Similarly, every Duckiedrone is given a name when first initialized. To refer to your Duckiedrone in this LX, replace `DUCKIEDRONE_NAME` in `DUCKIEDRONE_NAME.local` with its actual name. In the recorded example, `amelia.local` resolves (i.e., "translates") to `amelia`'s Wi-Fi address, as shown in [Figure 4](#figure-4).

<figure id="figure-4" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    amelia.local
      |
      | name lookup
      v
    192.168.1.201
      |
      | route selection
      v
    outgoing network interface
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 4: Name resolution and route selection for <code>amelia</code>.</figcaption>
</figure>

Names ending in `.local` normally use __multicast DNS (mDNS)__, a local name-resolution mechanism. [Notebook 21](./21-network-names-and-service-discovery.ipynb) explains how these names are resolved and how Duckietown device discovery uses the local network.

## Routing tables

The __routing table__ contains rules that associate destination addresses with an outgoing interface and, when needed, a gateway. This information helps answer, for example, which interface Linux selects when `amelia` needs to send data to `192.168.1.42` (the base station).

To inspect the local routing table, run:

```bash
ip route
```

The relevant routes in `amelia`'s captured output are:

The __Dynamic Host Configuration Protocol (DHCP)__ automatically provides network settings.

```shell
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
192.168.1.1 dev wlan0 proto dhcp scope link src 192.168.1.201 metric 600
```

Start with two entries: the route for the Wi-Fi subnet and the default route.

### Read the local route

The local route in `amelia`'s captured output is:

```shell
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
```

[Table 4](#table-4) explains the fields in this local route.

<table id="table-4" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 4: Fields in <code>amelia</code>'s local Wi-Fi route.</caption>
  <thead>
    <tr>
      <th>Field</th>
      <th>Meaning here</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>192.168.1.0/24</code></td>
      <td>Destination subnet</td>
    </tr>
    <tr>
      <td><code>dev wlan0</code></td>
      <td>Send through the Wi-Fi interface</td>
    </tr>
    <tr>
      <td><code>scope link</code></td>
      <td>Destinations are on the directly connected network</td>
    </tr>
    <tr>
      <td><code>src 192.168.1.201</code></td>
      <td>Preferred source address for packets <code>amelia</code> originates using this route</td>
    </tr>
  </tbody>
</table>

There is no `via` gateway in this entry. For the example destination `192.168.1.42`, `amelia` sends locally through `wlan0`.

"Locally" does not mean the Wi-Fi devices bypass the access point. It means the destination can be reached without routing the packet into another IP network.

### Read the default route

The default route in `amelia`'s captured output is:

```shell
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
```

This says: for destinations without a more specific route, send through `wlan0` to the gateway at `192.168.1.1`.

For an illustrative destination at `192.168.2.42`, [Figure 5](#figure-5) shows the first hop.

<figure id="figure-5" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    amelia (192.168.1.201)
      |
      v
    default gateway (192.168.1.1)
      |
      ?
      |
      v
    destination (192.168.2.42)
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 5: Default-route first hop for an off-subnet destination.</figcaption>
</figure>

While the routing table tells us which gateway `amelia` would use, it does not tell us whether that gateway has a working onward path to the destination, hence the `?`.

### Understand which entry wins

For ordinary routing in this table, Linux uses the __most specific matching destination prefix__. A route for one subnet is more specific than the default route.

Consequently, the default route does not take precedence only because it appears first in the output. [Table 5](#table-5) compares the route selected for each example destination.

<table id="table-5" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 5: Route selection examples from <code>amelia</code>'s table.</caption>
  <thead>
    <tr>
      <th>Example destination</th>
      <th>Matching route selected from <code>amelia</code>'s table</th>
      <th>First step</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>192.168.1.42</code></td>
      <td><code>192.168.1.0/24</code></td>
      <td>Directly over <code>wlan0</code></td>
    </tr>
    <tr>
      <td><code>192.168.2.42</code></td>
      <td><code>default</code></td>
      <td>Through <code>192.168.1.1</code> over <code>wlan0</code></td>
    </tr>
    <tr>
      <td><code>192.168.1.1</code></td>
      <td>The entry specifically for <code>192.168.1.1</code></td>
      <td>Directly over <code>wlan0</code> to the gateway itself</td>
    </tr>
  </tbody>
</table>

The remaining fields explain how the routes were installed or preferred. `proto kernel` marks a kernel-installed route; `proto dhcp` marks a route supplied automatically. A `metric` helps choose between otherwise comparable routes; lower values are preferred. These fields are documented in the [`ip route` manual](https://man7.org/linux/man-pages/man8/ip-route.8.html).

__Question:__ Suppose the default route disappeared, while the Wi-Fi interface and its local subnet route remained unchanged. Would `amelia` still have a route to the example base station at `192.168.1.42`?

<details>
<summary>Reveal answer</summary>

Yes. The `192.168.1.0/24` route still covers that destination. Losing the default route does not remove the local route.

`amelia` would, however, lack a route for destinations that previously depended on the default gateway.

</details>

A routing table describes forwarding decisions. It does not test whether another device responds. [Notebook 22](./22-network-diagnostics-and-testing.ipynb) combines route inspection with connection tests.

### Try it

From the authorized Duckiedrone shell established in [Notebook 13](./13-physical-duckiedrone-ssh-access.ipynb) or [Notebook 14](./14-virtual-duckiedrone-connections.ipynb), inspect the current network context without changing it:

```bash
hostname
ip -brief address
ip -brief link
ip route
```

Identify one active interface, its address and prefix, and the default gateway if the route table has one. Compare those observations with the recorded `amelia` example, but do not expect the names, addresses, or route metrics to match.

<details>
<summary>Check your result</summary>

The active interface is not necessarily named `wlan0`. A default route usually begins with `default via`; a directly connected subnet route normally has no `via` field. These commands only inspect the current context and do not test whether another device responds.

</details>

## Address families together

So far, we have followed IPv4 addresses. But `amelia`'s interface output also contained `::1`, and a hostname lookup on another system might return an address containing letters and colons.

These are __IPv6 addresses__. IPv6 uses 128-bit addresses, compared with IPv4's 32 bits, providing a much larger address space and addressing the exhaustion pressure that arose as internet use grew.

IPv6 writes addresses as hexadecimal groups separated by colons. A double colon, `::`, can replace one consecutive run of zero groups. For example, the documentation address `2001:db8:0:0:0:0:0:42` can be shortened to `2001:db8::42.`

[Table 6](#table-6) summarizes two useful IPv6 address forms.

<table id="table-6" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 6: Useful IPv6 address forms.</caption>
  <thead>
    <tr>
      <th>Address</th>
      <th>Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>::1</code></td>
      <td>IPv6 loopback: the current network environment</td>
    </tr>
    <tr>
      <td>An address beginning <code>fe80::</code></td>
      <td>A common form of IPv6 link-local address: usable on the local link and not forwarded by routers</td>
    </tr>
  </tbody>
</table>

The [IPv6 addressing specification](https://www.rfc-editor.org/rfc/rfc4291.html) defines these address forms.

A computer can use IPv4 and IPv6 together, often called __dual stack__. Having IPv4 connectivity does not establish that IPv6 connectivity is available as well.

Look again at `amelia`'s capture:

```shell
lo               UNKNOWN        127.0.0.1/8 ::1/128
wlan0            UP             192.168.1.201/24 metric 600
```

`amelia` has IPv6 loopback, but no IPv6 for Wi-Fi over `wlan0`, which is perfectly possible.

__Question:__ If another computer shows both an IPv4 address and a `fe80::` address on its Wi-Fi interface, does that prove it can reach the internet over IPv6?

<details>
<summary>Reveal answer</summary>

No. The `fe80::` address is link-local and cannot be forwarded across a router. It does not establish an IPv6 internet connection.

</details>

## Further reading

For more detail on inspecting assigned addresses, consult the Linux [`ip address` manual](https://man7.org/linux/man-pages/man8/ip-address.8.html).

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
